# 📓 Notebook 36 — Vector Databases & Agentic AI

The final notebook of Module 9 covers the *infrastructure* layer (vector databases) and the *behaviour* layer (agentic AI) — together they take your RAG POC from "one-PDF demo" to "actually useful AI system".

You'll leave this notebook able to: pick the right vector store for a job, write minimal pseudo-code to insert and query embeddings, explain how *tool calling* mechanically works, trace a ReAct loop, and build two final POCs end-to-end — a Chroma-backed semantic search and a command-line ReAct agent.

---

## 🎯 Learning objectives

1. **Explain** what makes a vector database different from SQL/NoSQL and when each is right.
2. **Compare** FAISS, Chroma, Qdrant, Weaviate, Pinecone, pgvector by deployment context.
3. **Insert and query** embeddings (with metadata filtering) using a few lines of pseudo-code.
4. **Define an agent** as LLM + Tools + Memory + Planning.
5. **Trace tool calling** end-to-end using the JSON-schema function-call pattern.
6. **Distinguish** ReAct, Plan-and-Execute, Tree-of-Thoughts, and Reflexion as control loops.
7. **Decide** when multi-agent pays for its coordination overhead.
8. **Build two POCs:** a semantic-product-search Chroma app and a ReAct command-line agent.

**Prerequisites:** NB 32 (LLM fundamentals), NB 35 (RAG deep dive), NB 33 (Copilot Agent setup).

**Time budget:** ~100 minutes including both POCs.


## Part A — Vector Databases

We met embeddings and FAISS in NB 35. This section steps back to ask: *when do you need a real vector database, and which one?*


## 1. Vector embedding — a refresher

An **embedding** is a learned function $f : X \to \mathbb{R}^d$ that maps an object (text, image, audio) into a $d$-dimensional vector such that *semantic similarity* corresponds to *geometric proximity*.

```
    dim 2
      ▲
      │           ● 'king'     ● 'queen'
      │              \           /
      │               ● 'monarch'    (semantic cluster)
      │
      │       ● 'apple'
      │       ● 'banana'        (different cluster)
      │
      └──────────────────────────► dim 1
```

Three distance metrics you'll encounter:

- **Cosine similarity** — angle-based, ignores magnitude. *Default for normalised text embeddings.*
- **Dot product** — cheap; equivalent to cosine when vectors are L2-normalised.
- **Euclidean (L2) distance** — cares about magnitude. *Less common for modern embeddings.*


## 2. Traditional DB vs vector DB

| | **Traditional (SQL/NoSQL)** | **Vector Database** |
|---|---|---|
| Primary lookup | Exact match on keys, ranges, joins | Approximate nearest neighbour (ANN) on vectors |
| Query language | SQL, MongoQL | Vector + optional metadata filter |
| Index type | B-tree, hash, inverted index | HNSW, IVF, IVF-PQ |
| Best for | Transactions, analytics on structured data | Semantic search, recommendation, RAG |
| Example query | `WHERE id = 42` | *"Find 5 chunks most similar to *this* query"* |

**They are complementary, not competing.** Modern stacks combine both: a SQL DB stores structured business data; a vector DB stores embeddings of unstructured text. **Hybrid search** (BM25 keyword + vector) is the production sweet-spot.


## 3. The vector-database landscape

| System | Type | When to choose it |
|---|---|---|
| **FAISS** | Local library (Meta) | Prototypes, in-process search, max throughput. No server, no metadata-rich filtering. |
| **Chroma** | Local / lightweight server | Developer-friendly Python API, great for tutorials, RAG demos, small-team apps. *Used in POC 1 below.* |
| **Qdrant** | Open-source server (Rust core) | Strong filtering, good performance, on-prem friendly. |
| **Weaviate** | Open-source server | Built-in modules (RAG, hybrid search, generative search). |
| **Pinecone** | Managed cloud | Production scale, low-latency SLAs, no infra to operate. |
| **pgvector** | Postgres extension | Re-use existing Postgres infra; transactional + vector in one DB. |

### Selection heuristic

- **Prototype** → FAISS or Chroma.
- **Self-hosted production** → Qdrant, Weaviate, or pgvector.
- **Managed at scale** → Pinecone.

> 💡 **Don't over-optimise the choice early.** Most production migrations between vector stores are 1–2 days of work — much smaller than the eventual investment in data quality, prompts, and evaluation. Start with the easiest option and switch when you have a concrete operational reason.


## 4. Inserting and querying — Chroma in 15 lines

**Chroma** is developer-friendly and persistent by default. Here's the full lifecycle:

```python
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Set up the client and a collection (persistent to disk)
client = chromadb.PersistentClient(path='./db')
coll = client.get_or_create_collection('docs')
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Insert documents with metadata
coll.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=model.encode(chunks).tolist(),
    metadatas=[{'source': s, 'page': p} for s, p in meta],
)

# 3. Query — with optional metadata filter
q_vec = model.encode(['How do I cancel an order?']).tolist()
results = coll.query(
    query_embeddings=q_vec,
    n_results=4,
    where={'source': 'policy.pdf'},  # metadata filter
)
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(round(dist, 3), doc[:120])
```

**What's different from raw FAISS:**

- **Persistence is built in.** Restart the app — the collection is still there.
- **IDs and documents** are stored alongside vectors. No separate metadata table.
- **`where={...}` clauses** let you filter at query time without rebuilding the index.
- **Multi-tenancy** — multiple named collections in one client.


## 5. 🛠️ POC 1 — Semantic Product Search with Chroma

**Scenario.** A retailer has a CSV catalog of ~2,000 products (title, description, category). Customers struggle with keyword search (*"cosy office sweater"* returns nothing). We want a small Streamlit app that finds items by *meaning*, not exact words, with a category filter in the sidebar.

**Goal.** A Streamlit app that ingests the CSV once, embeds each product description into a *persistent* Chroma collection, and serves a search box returning the top matches with similarity scores.


### POC 1 — Architecture

```
  Indexing (once, at startup if collection is empty):
    products.csv ─► MiniLM embeddings ─► Chroma collection (persisted ./db)

  Querying (every search):
    Streamlit search box ─► query embedding ─► top-k + category filter ─► ranked results
```

**Key property:** embedding the catalog runs *once*; Chroma persists vectors + metadata to disk. Second launch is instant.


### POC 1 — Agent-Mode prompt

```
Build a single-file Streamlit app app.py that performs semantic search over
a product catalog using Chroma as a persistent vector store. Create a helper
sample_data.py with a function get_products() that generates ~2,000
synthetic products (id, title, description, category, price) using
numpy/pandas and writes them to data/products.csv. On startup, if the
Chroma collection 'products' (path ./db) is empty, embed each product
description with sentence-transformers all-MiniLM-L6-v2 and add it with
metadata {category, price, title}; otherwise reuse the persisted
collection. The UI: a text input 'Search' and a sidebar st.selectbox
'Category' (with 'All'). On submit, embed the query, run
collection.query(n_results=10, where={...}) with the category filter, and
show a ranked table of title, category, price, and similarity score.
Provide requirements.txt (streamlit, chromadb, sentence-transformers,
pandas, numpy), .gitignore (.venv/, db/, data/), and a short README.
```

**Why this prompt produces *your* app:**

- **Persistence built in:** Chroma path `./db` — second launch is instant.
- **Demo data without an upload:** `sample_data.py` seeds the catalog.
- **Metadata filtering:** category stored as Chroma metadata, queried via `where={...}` — shows the difference vs pure FAISS.
- **Tunable surface:** sidebar selectbox + similarity threshold are obvious next iterations.


### POC 1 — Self-check

- [ ] First launch takes ~30 seconds (embedding 2,000 rows).
- [ ] Second launch starts in < 5 seconds (reused index).
- [ ] *"cosy office sweater"* returns plausible matches even if those exact words aren't in any title.
- [ ] The category filter actually narrows results.
- [ ] Similarity scores are sensible: top results are visibly more relevant than bottom results.

**If the first launch fails** (memory, model download), the most common cause is the `sentence-transformers` model download hanging behind a corporate proxy. Set `HF_HOME` to a local cache directory and retry.


## Part B — Agentic AI

RAG attaches *knowledge* to an LLM. **Agents attach *actions*** — letting the LLM call tools, observe the result, and iterate toward a goal. This section unpacks how that actually works mechanically, then we build a small ReAct agent end-to-end.


## 6. Defining an agent — LLM + Tools + Memory + Planning

```
                      ┌─────────────────────┐
                      │     Tools           │
                      │  APIs, functions,   │
                      │  retrievers         │
                      └─────────┬───────────┘
                                │
    ┌──────────────┐    ┌──────────────────┐    ┌─────────────────────┐
    │   Planning   │    │     LLM 'Brain'  │    │      Memory         │
    │ decompose    │ ◄──┤ reasoning,       ├─►  │ short-term &        │
    │ goals        │    │ decisions        │    │ long-term           │
    └──────────────┘    └──────────────────┘    └─────────────────────┘
```

**An agent in one sentence:** *an agent gets a natural-language goal, decides what to do, calls tools, remembers intermediate results, and iterates until the goal is reached or a stop condition fires.*

**The capability shift vs a plain chatbot:**

| Capability | Chatbot (LLM) | AI Agent |
|---|---|---|
| Answer questions | ✓ | ✓ |
| Execute actions | ✗ | ✓ (via tools) |
| Plan multiple steps | ✗ | ✓ |
| Use external systems (APIs, DBs) | ✗ | ✓ |


## 7. Quick primer — JSON and JSON Schema

**JSON** (JavaScript Object Notation) is a plain-text format for structured data. Two building blocks: an *Object* `{ "key": value, ... }` and an *Array* `[ value, ... ]`. Values are strings, numbers, booleans, `null`, or nested objects/arrays.

**A tiny JSON object:**

```json
{ "city": "Bielefeld", "units": "metric", "days": 3 }
```

**A JSON *Schema*** is itself a JSON document that describes *the shape valid input must have* — fields, types, required. The LLM reads it to know what it is *allowed* to emit; your code reads the LLM output knowing it should match the schema.

```json
{
  "type": "object",
  "properties": {
    "city":  { "type": "string" },
    "days":  { "type": "integer", "minimum": 1, "maximum": 14 }
  },
  "required": ["city", "days"]
}
```

**Why this matters for agents:** tool calling is *the LLM emitting JSON that conforms to a schema*. The schema is how you declare what tools the LLM can call.


## 8. 🧠 How tool calling actually works

**Step 1 — Declare tools as JSON-schema functions** (the same pattern across OpenAI, Anthropic, Gemini):

```python
tools = [{
    "name": "get_weather",
    "description": "Get current weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]
```

**Step 2 — The model returns a structured tool call** (not free text):

```json
{ "tool_use": { "name": "get_weather",
                 "input": { "city": "Bielefeld" } } }
```

**Step 3 — Your runtime executes the Python function** and feeds the result back into the model.

> 🎯 **Why this matters.** Tool calling turns the LLM from a text generator into a *controllable component of a software system.* The LLM *decides*; your code *executes*. You retain control over what's actually possible.


### Tool calling — end-to-end example

**Goal:** answer *"What's the weather in Bielefeld?"* using `get_weather`.

1. **User → LLM:** *"What's the weather in Bielefeld?"*
2. **LLM emits structured tool call:**
   ```json
   { "tool_use": { "name": "get_weather",
                    "input": { "city": "Bielefeld" } } }
   ```
3. **Runtime parses JSON, calls Python:** `result = get_weather("Bielefeld")  # "12°C, cloudy"`
4. **Runtime feeds result back:**
   ```json
   { "tool_result": { "name": "get_weather", "content": "12 C, cloudy" } }
   ```
5. **LLM:** *"It is currently 12°C and cloudy in Bielefeld."*

**Two key properties:**

- **Control stays in your code:** the LLM decides, you execute.
- **Loops naturally:** repeated tool calls are exactly what an *agent* does.


## 9. Memory — short-term vs long-term

| | **Short-term memory** | **Long-term memory** |
|---|---|---|
| What | Current conversation & scratchpad | Persistent facts, past sessions |
| Where | LLM context window | External store (vector DB, SQL) |
| Accessed | Always in the prompt | Retrieved on demand (via RAG!) |
| Lifetime | One session / loop | Across sessions and users |
| Failure mode | Context overflow → truncation | Stale or contradicting facts |

**Practical patterns:**

- **Rolling summary** — compress older turns once the context fills up.
- **Episodic memory** — store *"what happened"* per completed task (a vector DB of past task traces).
- **Profile memory** — inject user prefs at session start (*"this user prefers concise answers"*).

> 💡 **Long-term memory is RAG with the agent's own history as the corpus.** Same plumbing as NB 35 — different content.


## 10. Planning — task decomposition

Real-world goals (*"analyse Q3 churn and draft a memo"*) require a sequence of sub-tasks. Four common strategies:

| Strategy | Idea |
|---|---|
| **ReAct** | Interleave **Thought → Action → Observation** in one loop. *The simplest and most-used.* |
| **Plan-and-Execute** | First produce a multi-step plan, then execute (re-plan on failure). |
| **Tree-of-Thoughts** | Explore multiple reasoning paths in parallel, pick the best. |
| **Reflexion / Critic** | A second pass critiques the result and triggers a retry. |

**Stopping conditions matter.** Always cap an agent loop with:

- **Max iterations** — *"after 10 Thought→Action→Observation cycles, give up".*
- **Max tool calls** — *"no more than 20 tool calls per task".*
- **Token budget** — *"no more than 50,000 total tokens".*

Without these, an agent in a bad state will burn unbounded API credits. **This is the single most important guardrail.**


## 11. 🧠 The ReAct loop

The **ReAct** framework (Reason + Act) describes the working cycle of an agent (Yao et al., 2023):

```
    ┌──────────┐  create plan   ┌──────────┐
    │ Thought  │ ─────────────► │  Action  │
    │          │                │          │
    └──────────┘                └────┬─────┘
          ▲                          │
          │      evaluate result     │ execute tool
          │                          ▼
          │                    ┌────────────┐
          └────────────────────│ Observation│
                               └────────────┘
```

**The three steps:**

1. **Thought.** The agent analyses the goal and picks the next step.
2. **Action.** It executes a tool (e.g. an API call, a Python function).
3. **Observation.** It reads the result and plans the next step — until the goal is reached or the budget runs out.


### Worked example — a travel-planning agent

**Task:** *"Book a flight from Frankfurt to London for Tuesday and a hotel near the conference centre. Budget: 500 euros."*

| Step | Type | What the agent does |
|---|---|---|
| 1 | Thought | *I need to search for flights first.* |
| 1 | Action | `flight_search(FRA, LHR, Tuesday)` |
| 1 | Observation | *Cheapest flight: Airline X, 250 euros.* |
| 2 | Thought | *Flight fits the budget. Now search for a hotel.* |
| 2 | Action | `hotel_search(London, max_price=250)` |
| 2 | Observation | *Hotel GoodStay, 200 euros.* |
| 3 | Thought | *Flight (250) + Hotel (200) = 450 ≤ 500. Confirm.* |
| 3 | Action | `user_confirm(Flight X + GoodStay, 450 euros)` |
| 3 | Observation | *User confirms. Booking completed.* |

**What made it work:** each step is a clean *Thought → Action → Observation* cycle. The agent kept a running budget in its scratchpad memory; used `user_confirm` as a *guardrail* before any irreversible action; and the loop terminated as soon as the goal condition was met.


## 12. Multi-agent — when and when not

```
                 ┌─────────────────────┐
                 │  Orchestrator       │
                 │  (planner)          │
                 └──────────┬──────────┘
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
      ┌────────────┐ ┌────────────┐ ┌────────────┐
      │  Research  │ │  Analysis  │ │   Report   │
      │   Agent    │ │   Agent    │ │   Agent    │
      └────────────┘ └────────────┘ └────────────┘
```

**When multi-agent pays off:**

- **Clear role specialisation** — e.g. a *coder* agent and a separate *reviewer* agent with different prompts.
- **Parallel subtasks** that benefit from *independent* context windows.
- **Heterogeneous tool sets** that would clutter a single prompt.
- **Auditability** — separate logs per role make failures easier to attribute.

**When NOT to go multi-agent.** Multi-agent introduces coordination cost, more failure modes, and more tokens. For most tasks **a single well-prompted ReAct agent with good tools is enough — start there**.

> 🎯 **The trap.** Multi-agent looks impressive on a whiteboard. In practice, the bugs are between agents (one calls the other with the wrong schema, retries amplify), and the costs scale linearly with agent count. Adopt only when you've felt the pain of a single-agent design.


## 13. Example agent workflows

| Agent | Goal | Typical tools |
|---|---|---|
| **Research Agent** | Produce a literature briefing on topic *X* | `web_search`, `fetch_url`, `summarise`, `save_note` |
| **Data Analysis Agent** | Answer business questions over a CSV / DB | `run_sql`, `python_repl`, `plot`, `describe_schema` |
| **Procurement Agent** | Find best supplier for a part within budget | `rfq_db`, `supplier_api`, `cost_calculator`, `notify_buyer` |
| **Coding Agent** | Implement / fix a feature in a repo | `read_file`, `write_file`, `run_tests`, `git_diff` |

**Common pattern.** Each agent is: (a) a *system prompt* defining the role, (b) a *toolbox* of typed functions, (c) a *loop* running Thought → Action → Observation until done.

**Copilot Agent Mode (NB 33)** is precisely this pattern — a coding agent with tools `read_file`, `write_file`, `run_command`.


## 14. 🛠️ POC 2 — A ReAct command-line agent

**Use case.** A small command-line agent that, given a natural-language goal (e.g. *"What is the capital of France, times 2 letters?"*), **decides on its own** whether to look something up, perform a calculation, and when to stop — without us hard-coding the control flow.

**Goal.** A Python script `agent.py` that runs a ReAct loop over three tools (`search`, `calculator`, `final_answer`), uses an LLM as the brain, and prints each step so the reasoning is *auditable*.


### POC 2 — Architecture

```
                               ┌──────────────┐
                               │ calculator   │
                               └──────────────┘
  ┌──────────┐    ┌────────────┐
  │ User     │ ──►│ LLM Agent  │──►   ┌──────────────┐
  │ goal     │    │ (ReAct loop)│     │ search       │
  └──────────┘    └────────────┘      └──────────────┘
                                      ┌──────────────┐
                                      │ final_answer │
                                      └──────────────┘
```

**The loop in one sentence:** the LLM emits a strict *Thought / Action / Action Input* block, the runtime executes the named tool, feeds the observation back into the prompt, and repeats until `final_answer` is called or the step budget is exhausted.


### POC 2 — Agent-Mode prompt

```
Build a command-line ReAct agent in Python: agent.py plus tools.py. In
tools.py expose a dict TOOLS with three callables: calculator(expr) (safe
eval on arithmetic only), search(query) (lookup in a small in-memory
dictionary KB with at least 5 entries, e.g. 'capital of france' ->
'Paris'), and final_answer(text) (returns the text). In agent.py implement
run_agent(goal, max_steps=6): a loop that calls the Anthropic API
(claude-sonnet-4-6, key from ANTHROPIC_API_KEY) with a system prompt
forcing the format 'Thought:.../ Action:.../ Action Input:...'; parse the
response with regex; execute the named tool; append the Observation back
into the conversation history; stop when final_answer is invoked or
max_steps is reached. Print every step. At the bottom, an
if __name__ == '__main__' block that runs the agent on two example goals.
Provide requirements.txt (anthropic), .gitignore, and a README with the
run command.
```

**Why this prompt produces *your* agent:**

- **Strict output format** — the system prompt forces *Thought / Action / Action Input*, so a regex parser is reliable.
- **Three tools, one terminator** — `final_answer` is itself a tool, so the loop's stopping condition is *"the LLM called final_answer"*.
- **Hard step budget** — `max_steps=6` prevents runaway loops. *The single most important guardrail* for any agent that bills per token.
- **Files split by responsibility** — tools in `tools.py`, control flow in `agent.py`. Next iteration (*add a `rag_search` tool*) is a one-file change.


### POC 2 — Sample trace

Running the agent on *"What is the capital of France, times 2 letters?"* should produce something like:

```
[step 1]
  Thought: I need to find the capital of France first, then count its letters.
  Action: search
  Action Input: capital of france
  Observation: Paris

[step 2]
  Thought: Paris has 5 letters. The question asks for times 2 letters.
  Action: calculator
  Action Input: 5 * 2
  Observation: 10

[step 3]
  Thought: The answer is 10.
  Action: final_answer
  Action Input: 10
  Observation: 10

DONE. Final answer: 10
```

**Why this is so useful.** The trace is *human-readable*. If the agent makes a wrong decision, you can see *exactly* where — and adjust the system prompt or add a tool to fix it.


## 15. Synthesis — the three levels of LLM usage

```
    ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
    │  Just LLM        │    │  LLM + RAG       │    │  LLM + RAG +     │
    │                  │ ─► │                  │ ─► │  Agent           │
    │  Generates text  │    │  Grounded in     │    │  Plans and acts  │
    │  from frozen     │    │  current docs    │    │  via tools       │
    │  weights         │    │                  │    │                  │
    └──────────────────┘    └──────────────────┘    └──────────────────┘
       Hallucination          Precise, verifiable      Autonomous goal
       risk, no fresh         answers with             completion with
       data, no actions       citations                external systems
```

Module 9 has now walked the whole ladder:

- **NB 32** — what a Just-LLM is and isn't.
- **NB 33-34** — building real apps around LLMs.
- **NB 35** — adding the RAG layer (knowledge).
- **NB 36** — adding the agent layer (actions).

Each layer addresses the limitations of the previous one — and each layer is *optional*. Most production AI features in 2026 live at the RAG level; agents are a smaller (but rapidly growing) slice.


## 🧪 Practice exercises

### Exercise 1 — ⭐ Pick the vector store

For each scenario, pick a vector store from the §3 landscape and defend the choice in one sentence.

a. Weekend hackathon project to build a RAG over your team's wiki — 200 documents.
b. A 50-person company already running everything on Postgres; you want to add a semantic-search feature without introducing a new DB.
c. A B2B SaaS shipping a product to thousands of customers, each with their own private corpus, with an SLA for sub-100 ms latency.
d. A research project at university — you need *full control* over the index parameters and want to swap ANN algorithms freely.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**a. Chroma.** Local, persistent, no server to manage, perfect for the weekend timeframe. *FAISS is the alternative but lacks metadata filtering.*

**b. pgvector.** Re-uses the existing Postgres — no new operational surface, no separate DB to back up, transactional + vector in one query.

**c. Pinecone.** Managed, multi-tenancy, low-latency SLA. Self-hosted Qdrant/Weaviate could match the latency but would significantly grow the ops team — for a thousands-of-customers SaaS, paying for managed is usually worth it.

**d. FAISS** (with custom indices). Lowest-level library with the most knobs (HNSW, IVF, IVF-PQ, OPQ). For a research project where the *index algorithm is part of the experiment*, you want FAISS, not a database that hides the algorithm.
</details>

### Exercise 2 — ⭐ Read a tool-call

An LLM returns this tool-call JSON. Describe in plain English what the agent is trying to do and what your runtime should do next.

```json
{
  "tool_use": {
    "name": "send_email",
    "input": {
      "to": "customer@example.com",
      "subject": "Your order is on the way",
      "body": "Hi! Your order #8821 shipped today."
    }
  }
}
```

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**What the agent is trying to do:** call a `send_email` tool with three named arguments (recipient, subject, body). The LLM has *decided* it needs to send a shipping-confirmation email to a specific customer about a specific order.

**What your runtime should do next:**

1. **Parse and validate** the JSON against your `send_email` schema. Reject if required fields are missing or types are wrong.
2. **Apply guardrails BEFORE executing.** `send_email` is *irreversible* — you almost certainly want a *user confirmation step* or a *whitelist of allowed recipients* before any real send happens. The agent has no concept of "costly action" by default; *your runtime* is responsible for that.
3. **Execute** `send_email(to=..., subject=..., body=...)` in Python.
4. **Feed the result back** into the LLM's conversation history as a `tool_result` so it knows the send succeeded and can move to the next step.

**The wider lesson:** tool-call JSON is *just a request*. Whether it actually fires is your code's decision, and that decision is where most of the safety work lives.
</details>

### Exercise 3 — ⭐⭐ Trace a ReAct loop

Given the goal *"What is the current temperature in Bielefeld in Fahrenheit?"* and three tools `(get_weather_celsius(city), convert_c_to_f(c), final_answer(x))`, write the expected Thought / Action / Observation trace.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Expected trace:**

```
[step 1]
  Thought: I need the current temperature in Bielefeld; my get_weather tool
           returns Celsius, so I'll get that first.
  Action: get_weather_celsius
  Action Input: Bielefeld
  Observation: 12

[step 2]
  Thought: I have 12°C. The question asked for Fahrenheit, so I'll convert.
  Action: convert_c_to_f
  Action Input: 12
  Observation: 53.6

[step 3]
  Thought: The temperature in Fahrenheit is 53.6.
  Action: final_answer
  Action Input: 53.6°F
  Observation: 53.6°F

DONE.
```

**The key pattern:** the agent decomposes the problem into *the minimum number of tool calls*. No tool was called twice; no irrelevant tool (e.g. weather of a different city) was attempted. A *good* prompt biases the model toward minimal tool use; a *bad* prompt leaves it free to wander.
</details>

### Exercise 4 — ⭐⭐ Single agent vs multi-agent

For each scenario, decide if you'd build a *single ReAct agent* or *multiple coordinated agents*. Defend the choice in one sentence.

a. A documentation assistant that searches the company wiki and answers in natural language.
b. A research bot that (a) searches the web, (b) extracts data from PDFs, (c) writes a 5-page report, and (d) drafts an email.
c. A SQL-writing assistant that converts a natural-language question to SQL and runs it.
d. A *code review* system that (a) reads a diff, (b) runs tests, (c) suggests improvements, (d) checks against a style guide, and (e) generates a final summary.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**a. Single agent.** One role, one toolbox (search, read_doc). Multi-agent would add overhead with no clear separation of concerns.

**b. Multi-agent.** Four genuinely different *roles* — researcher (web), extractor (PDF), writer (report), email-drafter. Each benefits from its own system prompt and context window. The orchestrator coordinates.

**c. Single agent.** Two tools (text_to_sql, run_sql), one role, one short loop. Single-agent ReAct.

**d. Probably multi-agent.** Code-review benefits from a *coder/reviewer* split (one drafts, one critiques — Reflexion pattern). Style-guide checking could be a third specialised agent. The PR-summary generation is its own step.

**The pattern:** multi-agent pays off when there are *clearly different roles with different system prompts*. Multi-tool-but-one-role tasks stay single-agent.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

A teammate's ReAct agent is *looping* — it calls `search` 30 times in a row with slightly different queries, never gets to `final_answer`, and burns API credits. What's the most likely cause and what's the fix?

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Most likely cause:** the agent gets a partial / unsatisfactory result from `search`, decides *"let me try a slightly different query"*, gets another partial result, decides the same thing again, ad infinitum. The control loop has no incentive to commit to an answer — and crucially, no *step budget*.

**The fix has two parts:**

1. **Add a hard `max_steps` budget** (e.g. 6). When exceeded, the runtime forces `final_answer` with whatever the agent knows so far. This is *the single most important agent guardrail* — and the canonical mistake of every first-time agent builder.

2. **Strengthen the system prompt:** *"Use search at most 2 times. If the answer is not certain after 2 searches, return the best-guess final_answer with the caveat 'I'm not certain'."*

**The wider lesson:** an agent without a step budget is a runaway process by default. Cap iterations, cap tokens, cap tool calls. Even if you trust the model, the budget protects you from prompt-injection attacks that turn off the trust.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Build both POCs

Build POC 1 (Chroma semantic search) and POC 2 (ReAct agent) end-to-end. Submit:

1. Screenshots of both running.
2. Two *example traces* from POC 2 — one where the agent reaches `final_answer` cleanly, one where it hits the step budget.
3. A short reflection: *which of the two felt harder to make work, and why?*

<details>
<summary>💡 <b>Solution / Answer</b></summary>

No formal answer key. Typical findings:

- POC 1 (Chroma) is *easier than expected*. The persistence and metadata filtering work out of the box once you've used FAISS.
- POC 2 (ReAct) is *harder than expected*. The model deviates from the strict *Thought/Action/Action Input* format almost every other call; regex parsing is brittle. **The realistic fix is to use the provider's structured tool-calling API instead of free-text parsing** — and that's exactly the migration most agent frameworks (LangChain, LlamaIndex) handle for you.
- The *step-budget exhaustion* trace from POC 2 is uniquely valuable to keep — it shows you what "runaway" looks like in your specific setup. Save it for future reference.
</details>

### Stretch exercise B — ⭐⭐⭐ Add a RAG tool to your ReAct agent

Extend POC 2's agent with a fourth tool: `rag_search(question)`. Internally, this tool wraps the RAG POC from NB 35 — it embeds the question, retrieves top-4 chunks from a previously-built FAISS index over a PDF, and returns a single text string the agent can read.

Test the agent on a question that needs *both* the calculator *and* the PDF (e.g., the PDF says *"our quarterly target is 1200 units"*; the question is *"What's our daily target, given the quarterly target in the document?"*).

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Sketch of the new tool:**

```python
import faiss
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer('all-MiniLM-L6-v2')
_index = faiss.read_index('rag.index')  # pre-built
_chunks = json.load(open('chunks.json'))

def rag_search(question):
    q_vec = _model.encode([question])
    D, I = _index.search(q_vec, k=4)
    return ' '.join(_chunks[i] for i in I[0])
```

**Expected trace** for *"daily target from quarterly"*:

1. `rag_search("quarterly target")` → *"...our quarterly target is 1200 units..."*
2. `calculator("1200 / 90")` → 13.33
3. `final_answer("About 13 units per day, based on a 1200-unit quarterly target.")`

**The wider point:** an agent gains a *qualitatively new* capability for each tool you add. With RAG + calculator + final_answer, the agent can now answer *questions that require both fact lookup and arithmetic* — a class of question neither tool alone can handle.
</details>

### Stretch exercise C — ⭐⭐⭐ Compare ReAct vs Plan-and-Execute

On the same problem (your favourite mixed lookup + calculation question), run two versions of your agent:

**ReAct version** — system prompt forces *Thought / Action / Observation* per step.

**Plan-and-Execute version** — system prompt asks the model to first produce *the full plan* as a numbered list, then execute each step in order without re-planning.

Compare: (a) total tokens, (b) total wall-clock time, (c) success rate over 5 runs, (d) what each one does when a step fails.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Typical findings:**

- **Tokens.** Plan-and-Execute usually uses *fewer* tokens for the same task — the plan replaces multiple "Thought" preambles. On the other hand, the plan is often *wrong on the first try* and re-planning re-incurs the cost.
- **Wall-clock.** Plan-and-Execute has *fewer round-trips* to the LLM (one for the plan + one per execution) vs ReAct (one per step including Thought). Plan-and-Execute is usually faster.
- **Success rate.** ReAct is more *robust* to surprises — when an observation contradicts the plan, ReAct adapts; Plan-and-Execute often barrels through and fails. For our exercise, both should succeed.
- **When a step fails:** ReAct *reasons about the failure* and tries a different tool. Plan-and-Execute usually *retries the same step* or gives up.

**Practical recommendation:** *ReAct is the right default for production agents.* Plan-and-Execute is faster on well-defined deterministic tasks but brittle in the real world. Most real-world agent failures come from *unexpected observations*, and ReAct handles those better.
</details>

### Stretch exercise D — ⭐⭐⭐ Sketch a multi-agent system

Design (don't build) a multi-agent system for one of these scenarios. Specify: the *roles*, their *system prompts*, their *tools*, the *orchestrator's plan*, and the *message schema* between agents.

a. *Customer-support triage*: incoming email → categorise → answer or escalate → update CRM.
b. *Code review*: PR diff → static-analysis pass → AI review pass → reviewer notes → final summary.
c. *Weekly intelligence briefing*: news monitoring → relevance filtering → summary → email to leadership.
d. *Procurement*: RFQ document → supplier shortlist → cost compare → recommendation.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example for (a) customer-support triage:**

- **Roles:** *Classifier* (categorises emails), *Responder* (drafts replies for solved categories), *Escalator* (formats edge cases for human review), *CRM-updater* (writes to the CRM).
- **Tools per role:** Classifier → `classify(text)`. Responder → `rag_search(question)`, `draft_email(...)`. Escalator → `create_ticket(...)`. CRM-updater → `crm.update_case(...)`.
- **Orchestrator plan:** for each email → Classifier → branch: if category in {billing, password-reset, shipping-update} → Responder → CRM-updater; else → Escalator.
- **Message schema:**
  ```
  { "email_id": "...", "category": "...", "confidence": 0.91,
    "action_taken": "responded" | "escalated",
    "draft_reply": "..." (optional), "escalation_ticket_id": "..." (optional) }
  ```

**What makes the design solid:** every agent has a *narrow role* (a single system prompt fits comfortably); messages between them are *structured JSON* not free text (testable, parseable); the orchestrator's logic is *simple branching*, not another LLM call.

**The wider lesson:** good multi-agent designs look like *org charts* — clear roles, narrow responsibilities, structured handoffs. Bad multi-agent designs look like *committees* — everyone's discussing everything, no one's accountable.
</details>

## 🎁 Bonus mini-project — Ship a small agent that earns its keep

Pick one *real* personal or work task that currently takes you ≥ 30 minutes a week, and design a tiny agent for it. Examples:

- A weekly *email digest* of unread newsletters using a RAG over your archive.
- An *invoice extractor* that reads PDFs from a folder and writes a CSV.
- A *meeting-prep* agent that, given a calendar event, retrieves the relevant background from a Notion archive.

Build it through Copilot Agent Mode using the prompt patterns from NB 33 + this notebook's POCs. Measure: did it actually save you the 30 minutes?

**The point of this exercise is not the agent — it's *whether the agent earns its keep*. Most do not.** Treating the question seriously is what separates real AI engineers from people who built impressive demos.


## 🧠 Key takeaways

- **Embeddings + ANN** make semantic search practical at scale.
- **Vector databases** provide persistence, ANN indexing, and metadata filtering. Pick by *deployment context*: FAISS / Chroma for prototypes, Qdrant / Weaviate / pgvector for self-hosted, Pinecone for managed.
- **Agent = LLM + Tools + Memory + Planning.** Tool calling via JSON-schema is the primary mechanical interface.
- **ReAct** (Thought → Action → Observation) is the default control loop. Always cap iterations.
- **Multi-agent only when role specialisation clearly pays for the coordination overhead.** Default to single-agent.
- **The three levels of LLM usage:** Just-LLM → +RAG → +Agent. Each layer addresses the previous one's limitations.

## ✅ Self-assessment

- I can pick the right vector database for a given deployment context.
- I can draw the tool-calling flow end-to-end and explain why control stays with my code.
- I can trace a ReAct loop on paper for a multi-step problem.
- I built both POCs end-to-end and can explain how they extend the RAG POC of NB 35.
- I know to *always* cap an agent's iterations.

## 🚀 Where next

→ You've finished Module 9 — the deepest applied section of the course. Suggested next steps:

- **Module 7 Capstones** (`../07_capstones/`) — combine NB 33-36 into a single capstone.
- **Module 8 NB 31** (`../08_business_ai/31_bpm_governance_poc_mvp.ipynb`) — the methodology to take your POC to MVP and Production.
- **NB 22** (`../05_ai_engineering/22_ai_evaluation.ipynb`) — the evaluation discipline that turns demos into reliable systems.

Good luck. From here on, the limiting factor is not the technology — it's whether you can find a problem worth solving with it.
